In [1]:
import xml.etree.ElementTree as ET
import os

In [2]:
def build_sumo_detector_file(out_file, detector_output_file):
	# build detector data
	edges = ["e1_{}", "e2_{}", "e3_{}", "e4_{}", "e5_{}", "e6_{}", "er1_{}"]
	lanes = [6, 6, 7, 6, 6, 6, 1]
	types = ["source", "between", "between", "between", "between", "sink", "source"]

	# Create the root element
	root = ET.Element("additional")
	for i in range(len(edges)):
		det_type = types[i]
		for j in range(lanes[i]):
			lane_id = edges[i].format(j)
			detector_id = "det_{}".format(lane_id)
			inductionLoop = ET.SubElement(root, "inductionLoop")
			inductionLoop.set("id", detector_id)
			inductionLoop.set("lane", lane_id)
			inductionLoop.set("period", "60")
			inductionLoop.set("file", detector_output_file)
			if i == 0:
				inductionLoop.set("pos", "40")
			elif i == 6:
				inductionLoop.set("pos", "20")
			else:
				inductionLoop.set("pos", "0")

	# Create an ElementTree object from the root element
	tree = ET.ElementTree(root)
	ET.indent(tree, space="\t", level=0)

	# Write the tree to an XML file
	tree.write(out_file)  

In [3]:
build_sumo_detector_file("detector_test.add.xml", "./outputs/i80_detectors.out.xml")

In [ ]:
def parse_detector_output(detector_output_file):
	flow_dict = {}
	tree = ET.parse(detector_output_file)
	root = tree.getroot() 
	for child in root:
		if child.tag == "interval":
			detector_id = child.attrib["id"]
			det_root_id = detector_id.split("_")[0] + "_" + detector_id.split("_")[1]
			minute = int(float(child.attrib["begin"])/60.0)
			flow = int(float(child.attrib["flow"]))
			_key = "{}:{}".format(det_root_id, minute)
			flow_dict[_key] = flow_dict.get(_key, 0) + flow
	return flow_dict